# 🔍 Trace — SFT Warm-Start Training (Google Colab)

This notebook runs SFT (Supervised Fine-Tuning) on a **free T4 GPU** in Google Colab.

## Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your `sft_demos.jsonl` file when prompted

---

## Step 1: Install Dependencies
This installs Unsloth (optimized for Colab T4) + TRL + all required packages.

In [ ]:
%%capture
# Install Unsloth for Colab (handles torch/CUDA compatibility automatically)
!pip install unsloth
# Install TRL and other training dependencies
!pip install trl>=0.15.0 datasets>=2.18.0 peft>=0.10.0 pyyaml wandb

In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
    )

## Step 2: Upload Your SFT Dataset
Upload `data/sft_demos.jsonl` from your local `trace_project/trace/` folder.

In [ ]:
import os
from google.colab import files

# Create data directory
os.makedirs("data", exist_ok=True)

# Check if already uploaded
if os.path.exists("data/sft_demos.jsonl"):
    print("✅ Dataset already exists!")
else:
    print("📂 Upload your sft_demos.jsonl file...")
    uploaded = files.upload()
    # Move to data directory
    for name in uploaded:
        os.rename(name, f"data/{name}")
        print(f"✅ Saved to data/{name}")

# Verify
import json
with open("data/sft_demos.jsonl") as f:
    lines = [json.loads(l) for l in f if l.strip()]
print(f"\n📊 Dataset: {len(lines)} examples")
print(f"📊 First example preview: {lines[0]['text'][:200]}...")

## Step 3: Load Model with Unsloth
Loads Qwen2.5-3B-Instruct in 4-bit with Unsloth optimizations.

In [ ]:
from unsloth import FastLanguageModel

# ── Configuration (matches your grpo_config.yaml) ──────────────────
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

# LoRA config
LORA_R = 32
LORA_ALPHA = 32
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Model loaded with LoRA adapters!")
print(f"   Trainable params: {model.print_trainable_parameters()}")

## Step 4: Load Dataset

In [ ]:
from datasets import Dataset

sft_data = []
with open("data/sft_demos.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if line:
            sft_data.append(json.loads(line))

dataset = Dataset.from_list(sft_data)
print(f"✅ Dataset loaded: {len(dataset)} examples")
print(f"   Columns: {dataset.column_names}")

## Step 5: Configure & Run SFT Training
Uses TRL's SFTTrainer with the same config as your `grpo_config.yaml`.

In [ ]:
from trl import SFTTrainer, SFTConfig

# ── SFT config (matches your grpo_config.yaml sft section) ────────
OUTPUT_DIR = "./outputs/trace-sft"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,                    # from your config
    per_device_train_batch_size=4,          # from your config
    gradient_accumulation_steps=2,          # from your config
    learning_rate=2e-4,                     # from your config
    max_seq_length=2048,                    # from your config
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",                       # set to "wandb" if you want logging
    dataset_text_field="text",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print(f"\n{'='*60}")
print(f"[Trace] Starting SFT Training")
print(f"{'='*60}")
print(f"  Model:    {MODEL_NAME}")
print(f"  Examples: {len(dataset)}")
print(f"  Epochs:   2")
print(f"  Batch:    4 x 2 grad_accum = 8 effective")
print(f"  LR:       2e-4")
print(f"{'='*60}\n")

trainer.train()
print("\n✅ SFT Training Complete!")

## Step 6: Save Model

In [ ]:
# Save LoRA adapters (lightweight, ~50-100MB)
print("Saving LoRA adapters...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA saved to {OUTPUT_DIR}")

# Save merged 16-bit model (full model, ~6GB)
MERGED_DIR = "./outputs/trace-sft-merged"
print(f"\nSaving merged 16-bit model to {MERGED_DIR}...")
model.save_pretrained_merged(
    MERGED_DIR, tokenizer,
    save_method="merged_16bit",
)
print(f"✅ Merged model saved!")

## Step 7: Export GGUF for Ollama (Optional)
Creates a GGUF file you can load directly into Ollama on your local machine.

In [ ]:
# Export GGUF q4_k_m (good quality/size balance, ~2GB)
GGUF_DIR = "./outputs/trace-sft-gguf"
print("Exporting GGUF (q4_k_m)...")
try:
    model.save_pretrained_gguf(
        GGUF_DIR, tokenizer,
        quantization_method="q4_k_m",
    )
    print(f"✅ GGUF saved to {GGUF_DIR}")
except Exception as e:
    print(f"⚠️ GGUF export failed: {e}")
    print("This is optional — you still have the merged model above.")

## Step 8: Quick Test — Generate a Sample

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": (
        "You are an expert financial auditor. Accurately retrieve, verify, "
        "and aggregate transactions from digital sources.\n"
        "You MUST respond ONLY with valid JSON in this exact format:\n"
        '{"action_type": "PLAN|RETRIEVE|MEMORIZE|VERIFY|ANSWER", '
        '"content": "...", "source": "gmail|sheets|null"}\n'
        "Do NOT include any text outside the JSON object."
    )},
    {"role": "user", "content": (
        "[STEP 0]\n"
        "GOAL: Find all Uber ride receipts from Gmail in the last 30 days.\n"
        "SOURCES: gmail\n\n"
        "Choose your next action and respond with valid JSON only."
    )},
]

prompt = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=200, temperature=0.7, do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = response[len(tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)):]

print("\n" + "="*60)
print("[Trace] Test Generation:")
print("="*60)
print(response.strip())
print("="*60)

# Validate JSON
try:
    parsed = json.loads(response.strip())
    print(f"\n✅ Valid JSON! Action: {parsed.get('action_type')}")
except json.JSONDecodeError:
    print(f"\n⚠️ Not valid JSON — model may need more training")

## Step 9: Download Your Trained Model
Download the files to your local machine.

In [ ]:
import shutil

# Option A: Download LoRA adapters only (~50-100MB, recommended)
print("📦 Zipping LoRA adapters...")
shutil.make_archive("trace-sft-lora", "zip", OUTPUT_DIR)
files.download("trace-sft-lora.zip")
print("✅ Download started for LoRA adapters")

In [ ]:
# Option B: Download GGUF for Ollama (~2GB)
if os.path.exists(GGUF_DIR):
    gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
    if gguf_files:
        gguf_path = os.path.join(GGUF_DIR, gguf_files[0])
        print(f"📦 Downloading GGUF: {gguf_files[0]}")
        files.download(gguf_path)
    else:
        print("No GGUF file found")
else:
    print("GGUF not exported — run Step 7 first")

## Optional: Push to HuggingFace Hub
If you want to store the model on HuggingFace instead of downloading.

In [ ]:
# Uncomment and fill in to push to HuggingFace Hub
# HUB_REPO = "your-username/trace-sft-qwen2.5-3b"  # change this!
# HF_TOKEN = "hf_..."  # your HuggingFace token
#
# model.push_to_hub_merged(
#     HUB_REPO, tokenizer,
#     save_method="merged_16bit",
#     token=HF_TOKEN,
# )
# print(f"✅ Pushed to https://huggingface.co/{HUB_REPO}")